In [16]:
import torch
import glob
import os
import time
from torchvision.datasets import Food101, ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.models import ResNet18_Weights
from torchvision.models import EfficientNet_B0_Weights
import torchvision.models as models
import pandas as pd

MODEL_WEIGHTS_PATH = "model_weights"


if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")


def load_model_weights(model_path):
    try:
        state_dict = torch.load(model_path, map_location=device)
        print(f"Successfully loaded {model_path}")
        return state_dict["model_state"]
    except Exception as e:
        print(f"Error loading {model_path}: {e}")
        return None
    


def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    total_time = 0.0  

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if device.type == "cuda":
                torch.cuda.synchronize()

            start = time.time()
            outputs = model(images)
            if device.type == "cuda":
                torch.cuda.synchronize()
            end = time.time()

            total_time += (end - start)

            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += images.size(0)

    avg_loss = total_loss / total
    acc = correct / total
    time_per_image = total_time / total
    throughput = total / total_time

    return avg_loss, acc, time_per_image, throughput

In [ ]:

res_net_model_paths = glob.glob(os.path.join(MODEL_WEIGHTS_PATH, "final_checkpoint_resnet*.pt"))

res_net_model_paths = sorted(res_net_model_paths)


transform = ResNet18_Weights.DEFAULT.transforms()
criterion = torch.nn.CrossEntropyLoss()


root = "test_splits"

loaders = {}



for folder in os.listdir(root):
    path = os.path.join(root, folder)
    if not os.path.isdir(path):
        continue

    dataset = ImageFolder(root=path, transform=transform)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    loaders[folder] = loader

res1 = []

for model_path in res_net_model_paths:
    model = models.resnet18(weights=None)
    model.fc = torch.nn.Linear(model.fc.in_features, 101)  
    model.load_state_dict(load_model_weights(model_path))
    model.to(device)
    model.eval()

    for folder, loader in loaders.items():
        
        avg_loss, acc, time_per_image, throughput = validate(model, loader, criterion, device)
        print(
            f"Model: {model_path}, Folder: {folder}, "
            f"Val Loss: {avg_loss:.4f}, Val Acc: {acc:.4f}, "
            f"Time/img: {time_per_image:.6f}, Throughput: {throughput:.2f}"
            )

        res1.append((model_path, folder, avg_loss, acc, time_per_image, throughput))

# saving results to csv
df1 = pd.DataFrame(res1, columns=["model_path", "folder", "val_loss", "val_acc", "time_per_image", "throughput"])
df1.to_csv("res_net_results.csv", index=False)

In [ ]:
efficient_net_model_paths = glob.glob(os.path.join(MODEL_WEIGHTS_PATH, "final_checkpoint_efficientnet*.pt"))

efficient_net_model_paths = sorted(efficient_net_model_paths)



transform = EfficientNet_B0_Weights.DEFAULT.transforms()
criterion = torch.nn.CrossEntropyLoss()

root = "test_splits"

loaders = {}
for folder in os.listdir(root):
    path = os.path.join(root, folder)
    if not os.path.isdir(path):
        continue

    dataset = ImageFolder(root=path, transform=transform)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)
    loaders[folder] = loader

res2 = []

for model_path in efficient_net_model_paths:
    model = models.efficientnet_b0(weights=None)

    in_features = model.classifier[1].in_features
    model.classifier[1] = torch.nn.Linear(in_features, 101)

    model.load_state_dict(load_model_weights(model_path))
    model.to(device)
    model.eval()
    
    for folder, loader in loaders.items():
    
        avg_loss, acc, time_per_image, throughput = validate(model, loader, criterion, device)
        print(
            f"Model: {model_path}, Folder: {folder}, "
            f"Val Loss: {avg_loss:.4f}, Val Acc: {acc:.4f}, "
            f"Time/img: {time_per_image:.6f}, Throughput: {throughput:.2f}"
            )

        res2.append((model_path, folder, avg_loss, acc, time_per_image, throughput))

# saving results to csv
df2 = pd.DataFrame(res2, columns=["model_path", "folder", "val_loss", "val_acc", "time_per_image", "throughput"])
df2.to_csv("efficient_net_results.csv", index=False)

In [ ]:
res3 = res1 + res2
df3 = pd.DataFrame(res3, columns=["model_path", "folder", "val_loss", "val_acc", "time_per_image", "throughput"])
df3.to_csv("combined_results.csv", index=False)
